# Parkflächen umverteilen

## Daten aufbereiten

### Parkflächen ermitteln
Quelle: Flächenmehrzweckkarte (FMZK) der Stadt Wien (https://www.data.gv.at/katalog/dataset/7cf0da04-1f77-4321-929e-78172c74aa0b)  
Stand: 15. Oktober 2024

Die FMZK enthält fehlerhafte Einträge:

```geopandas.read_file("fmzk.gpkg")```

```GEOSException: IllegalArgumentException: point array must contain 0 or >1 elements```

Umweg über QGIS:
* FMZK Geopackage importieren
* Layer filtern nach ```"F_KLASSE_TEXT" = "Ruhender Verkehr"```
* Layer CRS auf *EPSG: 31256* setzen
* Gefilterte Layer mit CRS *EPSG:3857* und ohne Metadaten exportieren als Geopackage

In [ ]:
import geopandas as gpd

parkflaechen = gpd.read_file("parkflaechen_wien.gpkg")
parkflaechen.plot()

### Parkflächen in einzelne Parkplätze aufteilen

#### Largest interior rectangles finden
Parkflächen sind oft wenig rechtecksähnlich und können Löcher enthalten (z.B. Baumscheiben).  
Für die Aufteilung in einzelne Parkplätze zunächst die Parkflächen-Polygone mit möglichst großen Rechtecken füllen.  
Dafür gibts folgendes Package: https://github.com/OpenStitching/lir

Herausforderungen:
1. **Das LIR-Package findet nur axis-aligned LIRs**  
Parkflächen-Polygone sind willkürlich orientiert
2. **Das LIR-Package repräsentiert den Polygon intern als binary grid**  
Parkflächen-Polygone liegen in LAT/LON-Koordinaten, also Gleitkommazahlen vor. Skalierung auf Ganzzahlen erfordert zu großes grid.

Lösungen:
1. Axis-alignment
    1. Minimum-Bounding-Rectangle (MBR) für Parkflächen-Polygon berechnen
    2. Winkel zwischen MBR und x-Achse berechnen
    3. Parkflächen-Polygon um berechneten Winkel rotieren
    4. LIR berechnen und zurückrotieren um Schwerpunkt des Originalpolygons
2. Koordinatensystem
    1. LAT/LON Koordinaten des Polygon offsetten um Minimalkoordinaten
    2. Polygon als binary grid darstellen, Genauigkeit von 10cm (resolution=0.1) ausreichend
    3. LIR berechnen und Koordinaten wieder zurückskalieren und -offsetten

Gefundenes LIR von Parkflächen-Polygon abschneiden und Algorithmus erneut anwenden bis LIR kleiner als 2x4 bzw. 4x2 Meter (angenommene Mindestgröße eines einzelnen Parkplatzes).

In [34]:
import numpy as np
import math
from shapely.geometry import Polygon, box
from shapely import affinity
import cv2 as cv
import largestinteriorrectangle as lir

def rotate_polygon_to_axis(poly):
    """
    Rotates a polygon to be as axis-aligned as possible based on its minimum bounding rectangle.

    Parameters:
      poly (Polygon): A Shapely polygon.

    Returns:
      rotated_poly (Polygon): The polygon rotated to be axis-aligned.
      angle (float): The angle (in degrees) used for rotation.
      centroid (tuple): The centroid (x, y) of the original polygon before rotation.
    """
    # Get the minimum area bounding rectangle (already axis-aligned)
    min_rect = poly.minimum_rotated_rectangle

    # Get the coordinates of the bounding rectangle
    rect_coords = list(min_rect.exterior.coords)

    # Ensure we have at least 4 unique points
    if len(rect_coords) < 5:
        raise ValueError("Bounding rectangle does not have enough points")

    # Extract the first two distinct points to compute orientation
    x1, y1 = rect_coords[0]
    x2, y2 = rect_coords[1]

    # Compute the angle between the first edge and the x-axis
    angle = -math.degrees(math.atan2(y2 - y1, x2 - x1))

    # Rotate the polygon to align with the x-axis
    centroid = poly.centroid.coords[0]  # Store centroid before rotation
    rotated_poly = affinity.rotate(poly, angle, origin=centroid, use_radians=False)

    return rotated_poly, angle, centroid

def convert_polygon_to_grid(poly, resolution):
    """
    Converts a polygon into a binary grid representation.

    Parameters:
      poly (Polygon): A Shapely polygon.
      resolution (float): Grid cell size in meters.

    Returns:
      grid (2D ndarray): Binary grid representation of the polygon.
      offset (tuple): (minx, miny) used for translation.
    """
    minx, miny, maxx, maxy = poly.bounds

    # Define grid size
    width = int((maxx - minx) / resolution)
    height = int((maxy - miny) / resolution)

    # Create binary grid
    grid = np.zeros((height, width), dtype=bool)

    # Convert polygon to grid coordinates
    for i in range(height):
        for j in range(width):
            x = minx + j * resolution
            y = miny + i * resolution
            if poly.contains(Polygon([(x, y), (x+resolution, y), (x+resolution, y+resolution), (x, y+resolution)])):
                grid[i, j] = True

    return grid, (minx, miny)

def convert_grid_rect_to_3857_box(rect, offset, resolution):
    """
    Converts a rectangle from grid coordinates back to a Shapely box in EPSG:3857.

    Parameters:
      rect (array-like): [x, y, width, height] in grid cells.
      offset (tuple): (minx, miny) from grid conversion.
      resolution (float): Meters per grid cell.

    Returns:
      box: A Shapely box (Rectangle) in EPSG:3857.
    """
    x_grid, y_grid, width_grid, height_grid = rect
    minx, miny = offset

    # Convert grid coordinates to EPSG:3857 meters
    x_3857 = x_grid * resolution + minx
    y_3857 = y_grid * resolution + miny
    width_3857 = width_grid * resolution
    height_3857 = height_grid * resolution

    # Compute max x and y
    maxx = x_3857 + width_3857
    maxy = y_3857 + height_3857

    return box(x_3857, y_3857, maxx, maxy)

def compute_all_lirs(parking_lane_polygon, min_dim1=2, min_dim2=4, resolution=0.1):
    """
    Greedily extracts all Largest Interior Rectangles (LIRs) from the parking lane polygon.
    
    Only LIRs that satisfy a minimum dimension (at least 2x4 or 4x2 meters) are accepted.
    
    Parameters:
      parking_lane_polygon: The original parking lane (Shapely Polygon in EPSG:3857).
      min_dim1, min_dim2: The minimum required dimensions (in meters).
      resolution: Grid cell size in meters.
    
    Returns:
      A list of LIR polygons (in EPSG:3857) corresponding to parkable parts.
    """
    # Step 1: Rotate the polygon to be axis aligned.
    rotated_poly, angle, centroid = rotate_polygon_to_axis(parking_lane_polygon)
    
    current_poly = rotated_poly  # Work in rotated space
    found_lirs = []
    
    while True:
        # Step 2: Convert the current polygon to a grid.
        grid, offset = convert_polygon_to_grid(current_poly, resolution)
        
        # Step 3: Run the LIR algorithm on the grid. (Assuming lir.lir returns [x, y, w, h] in grid cells.)
        lir_grid_rect = lir.lir(grid)
        if lir_grid_rect is None:
            break  # No LIR found
        
        # Step 4: Convert the grid rectangle to a polygon in rotated coordinates.
        lir_poly_rotated = convert_grid_rect_to_3857_box(lir_grid_rect, offset, resolution)
        
        # Step 5: Check if the rectangle meets the minimum size criteria.
        lx, ly, ux, uy = lir_poly_rotated.bounds
        width = ux - lx
        height = uy - ly
        if not ((width >= min_dim1 and height >= min_dim2) or (width >= min_dim2 and height >= min_dim1)):
            break
        
        # Accept this LIR.
        found_lirs.append(lir_poly_rotated)
        
        # Step 6: Subtract the found LIR from the current polygon.
        new_poly = current_poly.difference(lir_poly_rotated)
        if new_poly.is_empty:
            break
        current_poly = new_poly
    
    # Step 7: Rotate all found LIRs back to the original coordinate system.
    final_lirs = [affinity.rotate(lir_poly, -angle, origin=centroid, use_radians=False) for lir_poly in found_lirs]
    return final_lirs

In [ ]:
import matplotlib.pyplot as plt

for index, row in parkflaechen.iterrows():
  parking_lane_polygon = row['geometry']
  lirs = compute_all_lirs(parking_lane_polygon, min_dim1=2, min_dim2=4, resolution=0.1)
  print(index)
#   x_lane, y_lane = zip(*parking_lane_polygon.exterior.coords)
#   plt.figure()
#   plt.axis("equal")
#   plt.plot(x_lane, y_lane, color="red")
#   for i, lir_poly in enumerate(lirs):
#         x, y = zip(*lir_poly.exterior.coords)
#         plt.plot(x, y, color="blue")
#   plt.show()
  

40687
49599
57159
7417
26542
10470
22542
57445
59538
62426
49125
62582


KeyboardInterrupt: 